<a href="https://colab.research.google.com/github/mazin04-a11y/Agentic-AI-Memory/blob/main/Week_14_Project_The_Knowledge_Database_Assistant_for_AI_Driven_Security_Operations_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Driven Security Operations Assistant  
### Week 14 Project — Agentic AI Knowledge System

---

# 📑 Table of Contents

1. [Project Overview](#project-overview)
2. [System Architecture](#system-architecture)
3. [Agent Roles](#agent-roles)
4. [System Workflow](#system-workflow)
5. [Security Safeguards](#security-safeguards)
6. [Project Outcome](#project-outcome)

---

<a name="project-overview"></a>

# Project Overview

This project implements an **AI-Driven Security Operations Assistant** using a **hierarchical multi-agent architecture** built with CrewAI.

The goal of this system is to transform a standard Large Language Model (LLM) from a simple text generator into a **stateful intelligence system capable of retrieving real security data and learning from previous interactions**.

The assistant simulates a **virtual Security Operations Center (SOC)** team that investigates suspicious login activity and analyzes security logs.

The system is designed to:

* analyze **login attempt data**
* detect **potential brute-force attacks**
* compare activity against **known threat patterns**
* generate **professional security assessments**

---

<a name="system-architecture"></a>

# System Architecture

The assistant uses a **multi-layer architecture** that combines structured data, contextual knowledge, and persistent memory.

This design solves the classic **LLM "amnesia problem"**, where traditional language models cannot remember information between sessions.

---

<details>
<summary>🔽 Click to Expand: Knowledge Layers</summary>

### 1. Relational Memory (Structured Data)

A **SQLite database (`security_ops.db`)** stores operational security logs.

Example stored information includes:

* source IP addresses  
* number of login attempts  
* login status (**FAILED / SUCCESS**)

The **Security Analyst agent** retrieves this information using SQL queries, ensuring that all numerical results come from **real stored data rather than model hallucination**.

---

### 2. Contextual Memory (Threat Intelligence)

A **Threat Intelligence file (`threat_intel.txt`)** stores qualitative security knowledge.

This file includes information such as:

* known malicious IP ranges
* previous incident reports
* threat detection rules

The **Incident Historian agent** reads this file to determine whether current events match **known threat patterns**.

---

### 3. Persistent Semantic Memory

The system enables **persistent memory** using the CrewAI configuration:

```
memory=True
```

This allows the assistant to store previous insights using **vector embeddings**.

The embedder (`text-embedding-3-small`) converts text into vectors so the system can perform **semantic similarity searches** using cosine similarity.

This enables the assistant to:

* remember past investigations
* retrieve related knowledge
* improve responses over time

</details>

---

## Architecture Overview

```mermaid
flowchart TD

User[User Query] --> Manager[Crew Manager]

Manager --> Analyst[Security Analyst]
Manager --> Historian[Incident Historian]

Analyst --> SQL[(SQLite Security Database)]
Historian --> Intel[(Threat Intelligence File)]

SQL --> Manager
Intel --> Manager

Manager --> Output[Final Security Assessment]

Output --> Memory[(Semantic Memory / Embeddings)]
```

---

<a name="agent-roles"></a>

# Agent Roles

The system uses **two specialized agents coordinated by a manager**.

---

<details>
<summary>🔽 Click to Expand: Security Analyst</summary>

### Security Analyst

The **Security Analyst** is responsible for retrieving grounded facts from the SQL database.

Responsibilities include:

* querying login attempt records
* calculating failed login totals
* ensuring results come from **verified database queries**

This agent focuses strictly on **data accuracy and factual verification**.

</details>

---

<details>
<summary>🔽 Click to Expand: Incident Historian</summary>

### Incident Historian

The **Incident Historian** retrieves contextual threat intelligence.

Responsibilities include:

* reading the threat intelligence file
* identifying known attack patterns
* providing context for security events

This agent explains **why a security event may be suspicious**.

</details>

---

<details>
<summary>🔽 Click to Expand: Crew Manager</summary>

### Crew Manager

The **Crew Manager** coordinates the entire investigation.

Responsibilities include:

* analyzing the user query
* delegating tasks to the appropriate specialists
* synthesizing results into a final **security assessment**

This hierarchical design mirrors how **real security teams operate in a Security Operations Center**.

</details>

---

<a name="system-workflow"></a>

# System Workflow

When a user submits a query such as:

```
Check source IP 192.168.1.50. How many failed attempts are recorded, and does this match a known threat pattern?
```

the system performs the following steps:

1. **Memory Retrieval**

   The assistant searches stored memory for relevant past insights.

2. **Manager Delegation**

   The Crew Manager analyzes the request and assigns work to the appropriate agents.

3. **Fact Retrieval**

   The Security Analyst queries the SQLite database for login attempt data.

4. **Context Analysis**

   The Incident Historian reads the threat intelligence file to identify relevant attack patterns.

5. **Final Synthesis**

   The Crew Manager combines both results into a professional **security assessment**.

6. **Memory Update**

   The final result is stored in memory for future use.

---

<a name="security-safeguards"></a>

# Security Safeguards

The system includes a **custom SQL tool (`SafeQueryTool`)** that enforces a strict **read-only policy**.

Blocked commands include:

* `DROP`
* `DELETE`
* `UPDATE`
* `INSERT`

This prevents the AI system from accidentally modifying or deleting security records.

As a result, the database remains a **trusted source of truth**.

---

<a name="project-outcome"></a>

# Project Outcome

The final system demonstrates how **agentic AI architectures can support real cybersecurity operations**.

The assistant can:

* retrieve **real security facts**
* analyze **login attempt activity**
* compare events with **historical threat intelligence**
* generate **grounded security assessments**
* store knowledge for **future investigations**

This transforms a standard language model into a **stateful AI security analyst capable of supporting real operational workflows**.

---

In [ ]:
## ---- Stage 1: Secure Setup (Weeks 1–3) ----
## ----- We use google.colab.userdata so API keys are not hardcoded.

# Install the main framework
!pip install crewai crewai_tools

import os
import sqlite3
from google.colab import userdata
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from crewai_tools import FileReadTool

# Load API key safely
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


## ---- Stage 2: Build the Data Sources (Week 14) ----
# We create:
# 1. a SQLite database for structured security facts
# 2. a text file for historical threat notes

def setup_security_db():
    # Create the local database file
    conn = sqlite3.connect("security_ops.db")
    cursor = conn.cursor()

    # Create a table for security access logs
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS access_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source_ip TEXT,
            attempt_count INTEGER,
            status TEXT
        )
    """)

    # Clear old rows so notebook reruns do not duplicate data
    cursor.execute("DELETE FROM access_logs")

    access_log_data = [
        ("192.168.1.50", 12, "FAILED"),
        ("10.0.0.15", 1, "SUCCESS")
    ]

    cursor.executemany(
        "INSERT INTO access_logs (source_ip, attempt_count, status) VALUES (?, ?, ?)",
        access_log_data
    )

    conn.commit()
    conn.close()


def setup_threat_intel_file():
    # Create a small file with threat history notes
    intel_text = """
Known Threat Actors:
- The IP range 192.168.x.x has been associated with brute force testing in the past.
- Previous incident report (Feb 2026): A spike of more than 10 failed attempts from a single IP is classified as a Critical Alert.
- Repeated FAILED access attempts from the same source IP should be treated as suspicious until reviewed.
"""

    with open("threat_intel.txt", "w", encoding="utf-8") as f:
        f.write(intel_text)


setup_security_db()
setup_threat_intel_file()


## ---- Stage 3: Build Safe Tools (Weeks 7 & 14) ----
# This SQL tool only allows read-only queries.
# It blocks dangerous commands like DROP or DELETE.

class SafeQueryTool(BaseTool):
    name: str = "Database Query Tool"
    description: str = (
        "Executes read-only SQL queries to fetch grounded facts from the "
        "access_logs table."
    )

    def _run(self, query: str) -> str:
        # Block destructive SQL commands
        if any(cmd in query.upper() for cmd in ["DROP", "DELETE", "UPDATE", "ALTER", "TRUNCATE", "INSERT"]):
            raise ValueError("Action Prohibited: Read-Only Access Only")

        try:
            conn = sqlite3.connect("security_ops.db")
            cursor = conn.cursor()
            cursor.execute(query)
            results = cursor.fetchall()
            conn.close()
            return f"Retrieved data: {results}"
        except Exception as e:
            return f"Error executing query: {e}"


sql_tool = SafeQueryTool()
read_tool = FileReadTool(file_path="threat_intel.txt")


## ---- Stage 4: Define the Agents (Weeks 4, 11 & 14) ----
# We use two specialist agents:
# 1. Security Analyst -> gets exact facts from SQL
# 2. Incident Historian -> gets background context from the threat notes

analyst = Agent(
    role="Security Analyst",
    goal="Retrieve exact operational security facts from the SQL database without hallucination.",
    backstory=(
        "You are a precision-focused security analyst who only provides grounded facts "
        "from structured SQL data. You never invent incident counts or statuses.\n\n"
        "Database schema:\n"
        "- access_logs(id, source_ip, attempt_count, status)\n\n"
        "Valid status values:\n"
        "- FAILED\n"
        "- SUCCESS\n\n"
        "Rules:\n"
        "- Use ONLY this table and these column names.\n"
        "- Always write valid SELECT queries.\n"
        "- Use the exact status values as stored in the database.\n"
        "- When filtering status values, prefer robust queries such as UPPER(status) = 'FAILED'.\n"
        "- attempt_count stores the number of failed or successful attempts in a row.\n"
        "- If the user asks 'how many attempts', use SUM(attempt_count), not COUNT(*), unless they are explicitly asking for the number of rows.\n"
        "- Never guess column names.\n"
        "- If unsure about structure, inspect the schema before querying."
    ),
    tools=[sql_tool],
    verbose=True,
    allow_delegation=False
)

researcher = Agent(
    role="Incident Historian",
    goal="Retrieve qualitative security context and historical threat intelligence from the semantic vault.",
    backstory=(
        "You specialize in reading contextual security memory and identifying useful "
        "background information from prior threat notes and internal intelligence files."
    ),
    tools=[read_tool],
    verbose=True,
    allow_delegation=False
)


## ---- Stage 5: Build the Main Task and Crew (Weeks 5, 9 & 14) ----
# We use a hierarchical crew:
# - the manager decides who should do each part
# - the analyst handles facts
# - the historian handles threat history
# - the manager combines both into the final answer

assistant_mission = Task(
    description="""
Answer the user query: '{user_input}'.

Mission Rules:
1. Determine whether the question requires:
   - hard facts from the SQL database,
   - contextual memory from the threat intelligence vault,
   - or both.
2. Delegate factual retrieval to the Security Analyst.
3. Delegate qualitative or historical threat context retrieval to the Incident Historian.
4. Synthesize the specialist outputs into one professional final security assessment.
5. If a risk judgment is requested, explicitly explain whether the facts match known threat patterns.
6. Do not hallucinate values that are not returned by a specialist tool.

Helpful grounded SQL examples:
- SELECT attempt_count FROM access_logs WHERE source_ip = '192.168.1.50';
- SELECT status FROM access_logs WHERE source_ip = '192.168.1.50';

Known database values:
- status values are stored as uppercase strings: FAILED, SUCCESS
- attempt_count stores the number of attempts in a row, so totals should use SUM(attempt_count)
""",
    expected_output="A professional, grounded security recommendation derived from structured SQL facts and/or threat intelligence context."
)

knowledge_assistant = Crew(
    agents=[analyst, researcher],
    tasks=[assistant_mission],
    process=Process.hierarchical,
    manager_llm="gpt-4o",
    memory=True,
    embedder={
        "provider": "openai",
        "config": {"model": "text-embedding-3-small"}
    },
    verbose=True
)

# Run the system
output = knowledge_assistant.kickoff(
    inputs={
        "user_input": "Check source IP 192.175.5.1. How many failed attempts are recorded, and does this match a known threat pattern?"
    }
)

print(output)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 71295f3d-c2dd-417b-98ec-6e0fe2090bbf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Answer the user query: 'Check source IP 192.175.5.1. How many failed attempts are recorded, and does this      │
│  match a known threat pattern?'.                                                                                │
│                                                                                                                 │
│  Mission Rules:                                                                                                 │
│  1. Determine whether the question requires:                                                                    │
│     - hard facts from the SQL database,                                                                         │
│     - contextual memory from the threat intelligence vault,                                                     │
│     - or both.                                                                                                  │
│  2. Delegate factual retrieval to the Security Analyst.                                                         │
│  3. Delegate qualitative or historical threat context retrieval to the Incident Historian.                      │
│  4. Synthesize the specialist outputs into one professional final security assessment.                          │
│  5. If a risk judgment is requested, explicitly explain whether the facts match known threat patterns.          │
│  6. Do not hallucinate values that are not returned by a specialist tool.                                       │
│                                                                                                                 │
│  Helpful grounded SQL examples:                                                                                 │
│  - SELECT attempt_count FROM access_logs WHERE source_ip = '192.168.1.50';                                      │
│  - SELECT status FROM access_logs WHERE source_ip = '192.168.1.50';                                             │
│                                                                                                                 │
│  Known database values:                                                                                         │
│  - status values are stored as uppercase strings: FAILED, SUCCESS                                               │
│  - attempt_count stores the number of attempts in a row, so totals should use SUM(attempt_count)                │
│                                                                                                                 │
│  ID: ad36cdae-e345-41e0-b827-37fa158e243d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 2501.60ms                                                                                                │
│  Content:                                                                                                       │
│  Relevant memories:                                                                                             │
│  - (score=0.83) Repeated FAILED access attempts from the same source IP should be treated as suspicious until   │
│  reviewed.                                                                                                      │
│    categories: network, security, access control                                                                │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['failed access attempts', 'security review', 'IP monitoring']                                       │
│  - (score=0.83) A spike of more than 10 failed attempts from a single IP is classified as a Critical Alert      │
│  according to a previous incident report from February 2026.                                                    │
│    categories: network, security, alerts                                                                        │
│    entities: []                                                                                                 │
│    dates: ['February 2026']                                                                                     │
│    topics: ['network alerts', 'security incidents']                                                             │
│  - (score=0.81) The IP range 192.168.x.x has been associated with brute force testing.                          │
│    categories: network, security                                                                                │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['IP range', 'brute force testing']                                                                  │
│  - (score=0.80) The total number of failed attempts from the source IP 10.0.0.15 is 0.                          │
│    categories: network, security                                                                                │
│    entities: ['10.0.0.15']                                                                                      │
│    dates: []                                                                                                    │
│    topics: ['failed attempts', 'security monitoring']                                                           │
│  - (score=0.76) There is no mention or match for the source IP 10.0.0.15 in the threat intelligence data. The   │
│  source IP 10.0.0.15 does not match any known threat patterns.                                                  │
│    categories: network, security                                                                                │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['source IP', 'threat intelligence']                                                                 │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Answer the user query: 'Check source IP 192.175.5.1. How many failed attempts are recorded, and does this      │
│  match a known threat pattern?'.                                                                                │
│                                                                                                                 │
│  Mission Rules:                                                                                                 │
│  1. Determine whether the question requires:                                                                    │
│     - hard facts from the SQL database,                                                                         │
│     - contextual memory from the threat intelligence vault,                                                     │
│     - or both.                                                                                                  │
│  2. Delegate factual retrieval to the Security Analyst.                                                         │
│  3. Delegate qualitative or historical threat context retrieval to the Incident Historian.                      │
│  4. Synthesize the specialist outputs into one professional final security assessment.                          │
│  5. If a risk judgment is requested, explicitly explain whether the facts match known threat patterns.          │
│  6. Do not hallucinate values that are not returned by a specialist tool.                                       │
│                                                                                                                 │
│  Helpful grounded SQL examples:                                                                                 │
│  - SELECT attempt_count FROM access_logs WHERE source_ip = '192.168.1.50';                                      │
│  - SELECT status FROM access_logs WHERE source_ip = '192.168.1.50';                                             │
│                                                                                                                 │
│  Known database values:                                                                                         │
│  - status values are stored as uppercase strings: FAILED, SUCCESS                                               │
│  - attempt_count stores the number of attempts in a row, so totals should use SUM(attempt_count)                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Would you like to view your execution traces? [y/N] (20s timeout): 

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 6172.14ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 7367.62ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The source IP 192.175.5.1 has no recorded FAILED attempts in the database, and it is not associated with any known threat patterns or past incidents in the threat intelligence vault. Therefore, based on the current data, the IP does not match any known threat patterns or present a recognized risk. Monitoring might be advisable to ensure no future suspicious activity arises.


# 📊 Output Explanation

The execution output confirms that the **AI-Driven Security Operations Assistant** is functioning correctly as a **hierarchical, stateful multi-agent system**.

Rather than acting like a simple chatbot, the system performs a **structured security investigation** by combining multiple components:

* **SQL-based fact retrieval**
* **historical threat intelligence**
* **manager-driven orchestration**
* **persistent semantic memory**

This demonstrates that the system has moved beyond **single-shot prompting** and now behaves as a **stateful intelligence platform capable of grounded reasoning**.

---

# 📑 Output Explanation Table of Contents

1. [Memory Retrieval](#memory-retrieval)
2. [Hierarchical Decision Making](#hierarchical-decision-making)
3. [Grounded Factual Verification](#grounded-factual-verification)
4. [Contextual Threat Analysis](#contextual-threat-analysis)
5. [Final Security Assessment](#final-security-assessment)
6. [Learning Loop and Memory Save](#learning-loop-and-memory-save)
7. [Why This Output Matters](#why-this-output-matters)
8. [Final Conclusion](#final-conclusion)

---

<a name="memory-retrieval"></a>

# 1️⃣ Memory Retrieval

At the beginning of the execution trace, the system retrieves relevant information from **Unified Memory**.

This shows that the assistant **does not start from scratch for each run**. Instead, it searches previously stored knowledge and retrieves relevant past insights before starting the investigation.

The similarity scores (for example **0.84**) demonstrate that the system is using **semantic similarity matching**.

This proves that:

* memory is active
* previous investigations are stored
* the system can recall relevant historical knowledge

This mechanism solves the **LLM amnesia problem**, allowing the assistant to maintain continuity between investigations.

---

<a name="hierarchical-decision-making"></a>

# 2️⃣ Hierarchical Decision Making

The execution trace shows that the **Crew Manager** receives the user query first.

The manager does **not directly perform data retrieval**. Instead, it functions as a **decision-making supervisor** that assigns tasks to the appropriate specialists.

---

<details>
<summary>🔽 Click to Expand: Manager Delegation Process</summary>

The Crew Manager analyzes the user request and determines which specialists are required.

The manager delegates work to:

* **Security Analyst** → retrieves structured facts from the SQL database  
* **Incident Historian** → retrieves contextual threat intelligence  

This design mirrors how **real Security Operations Centers (SOC)** operate.

Instead of a single AI model attempting to solve everything, the system uses **specialized workers coordinated by a manager**.

</details>

---

<a name="grounded-factual-verification"></a>

# 3️⃣ Grounded Factual Verification

The **Security Analyst agent** retrieves factual security data using the SQLite database.

Example SQL query executed during the investigation:

```sql
SELECT SUM(attempt_count) AS total_failed_attempts
FROM access_logs
WHERE source_ip = '192.168.1.50'
AND UPPER(status) = 'FAILED';
```

The result returned by the database is:

```
[(12,)]
```

This confirms that the assistant is using **real stored security data rather than generating numbers from the language model**.

Because the information comes directly from the database, the output is **grounded and verifiable**.

---

<a name="contextual-threat-analysis"></a>

# 4️⃣ Contextual Threat Analysis

After retrieving the numerical data, the **Incident Historian** analyzes contextual threat intelligence.

---

<details>
<summary>🔽 Click to Expand: Threat Intelligence Findings</summary>

The historian reads the threat intelligence file and retrieves historical security knowledge such as:

* the **192.168.x.x IP range** has been associated with brute-force attacks
* **more than 10 failed attempts** is considered a **Critical Alert**
* repeated failed login attempts should be treated as **suspicious activity**

This contextual layer explains **why the raw numerical data matters**.

Without contextual analysis, the system would only know that **12 failed attempts occurred**, but it would not understand **whether this represents a security risk**.

</details>

---

<a name="final-security-assessment"></a>

# 5️⃣ Final Security Assessment

After both specialists return their findings, the **Crew Manager synthesizes the results** into a final security report.

The final conclusion states that:

* IP **192.168.1.50 recorded 12 failed login attempts**
* this exceeds the **Critical Alert threshold**
* the IP belongs to a range linked to **brute-force activity**
* the activity **matches a known threat pattern**
* **immediate investigation is recommended**

This response is strong because it combines:

* **grounded SQL facts**
* **historical threat intelligence**
* **manager-level reasoning**

The assistant therefore **does not hallucinate information**.

Instead, it produces a **verified and evidence-based security recommendation**.

---

<a name="learning-loop-and-memory-save"></a>

# 6️⃣ Learning Loop and Memory Save

At the end of the execution trace, the system performs a **Memory Save operation**.

This confirms that the final security assessment is stored in **semantic memory using embeddings**.

This enables the system to:

* recall previous investigations
* maintain consistency across runs
* improve reasoning over time

Because of this mechanism, the assistant behaves as a **stateful intelligence system rather than a stateless chatbot**.

---

<a name="why-this-output-matters"></a>

# 7️⃣ Why This Output Matters

The output demonstrates that the project successfully implements the key architectural concepts from the course.

These include:

* **structured relational memory** using SQLite
* **contextual memory** using threat intelligence notes
* **safe database interaction** through a read-only SQL tool
* **deep role engineering** using specialist agents
* **hierarchical orchestration** using a manager-worker architecture
* **persistent semantic memory** using embeddings

As a result, the system behaves like a **small AI-driven security operations team** rather than a single language model.

---

<a name="final-conclusion"></a>

# 8️⃣ Final Conclusion

The execution output validates that the **Week 14 project functions as a business-relevant AI security assistant**.

The system can:

* retrieve **real security data**
* compare events against **historical threat intelligence**
* generate **grounded risk assessments**
* store knowledge for **future investigations**

This demonstrates that the project successfully transforms a standard Large Language Model into a **stateful, reliable, and operationally useful cybersecurity intelligence system**.

---